# Lesson 5: Human in the Loop

Note: This notebook is running in a later version of langgraph that it was filmed with. The later version has a couple of key additions:
- Additional state information is stored to memory and displayed when using `get_state()` or `get_state_history()`.
- State is additionally stored every state transition while previously it was stored at an interrupt or at the end.
These change the command output slightly, but are a useful addtion to the information available.


注意：此笔记本运行在录制时使用的较新版本的 langgraph 中。较新版本增加了一些关键功能：
- 附加状态信息存储在内存中，并在使用 `get_state()` 或 `get_state_history()` 时显示。
- 状态现在会在每次状态转换时额外存储，而之前状态是在中断时或结束时存储的。
这些操作会稍微改变命令输出，但可以作为可用信息的有用补充。

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [ ]:
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_tavily import TavilySearch
from uuid import uuid4

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 建立内存数据库连接，用于保存agent状态
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

In [3]:
"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.

在之前的示例中，我们已经对 `messages` 状态键进行了注释。
使用默认的 `operator.add` 或 `+` reducer，它总是
将新消息追加到现有消息数组的末尾。

现在，为了支持替换现有消息，我们对消息进行注释。
`messages` 键与一个自定义 reducer 函数一起使用，该函数替换了
具有相同 `id` 的消息，否则附加它们。
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    """
    自定义消息合并函数，支持替换已有消息
    
    :param left: 现有消息列表
    :param right: 新消息列表
    :return: 合并后的消息列表
    """
    # 为没有ID的消息分配UUID
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # 合并新旧消息
    # merge the new messages with the existing messages
    merged = left.copy()   # 创建现有消息的副本
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            # 检查是否已有相同ID的消息
            if existing.id == message.id:
                # 替换已有消息
                merged[i] = message
                break
        else:
             # 如果没有相同ID的消息，则追加新消息
            # append any new messages to the end
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [4]:
tool = TavilySearch(max_results=2)

## Manual human approval

In [5]:
class Agent:
    """
    支持人工干预的Agent，可在工具调用前暂停并允许人工审核
    """
    def __init__(self, model, tools, system="", checkpointer=None):
        """
        初始化Agent
        
        :param model: 语言模型（如ChatOpenAI）
        :param tools: 工具列表（如[TavilySearchResults]）
        :param system: 系统提示词
        :param checkpointer: 状态检查点（用于持久化）
        """
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        # 编译图，启用状态持久化和**关键设置：在action前中断**
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]  # 在执行工具前暂停
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [6]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatTongyi(
    model="qwen-max",  # 或其他通义千问模型
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),  # 通义千问 API key
    temperature=0.1, 
    streaming=True
)
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:

# 示例使用1：发送消息并自动处理
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
# for event in abot.graph.invoke({"messages": messages}, thread):
#     for v in event.values():
#         print(v)
# stream() 完整遍历会忽略中断
# invoke() 会在中断点停止并返回
# 只有在图真正暂停时，next 才会显示待执行的节点
result = abot.graph.invoke({"messages": messages}, thread)


{'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='518f2156-6cc3-4349-8f7d-4a12710c1c29'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in San Francisco"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '43a805b4-7ad2-41bd-9758-88d1e451e88f', 'token_usage': {'input_tokens': 1954, 'output_tokens': 22, 'total_tokens': 1976, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dca8932d-a32f-47cf-a0c6-1586a0f08cd0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'tool_call'}])]}


In [8]:
# 检查当前状态
# get_state()返回什么？
# - values: 当前状态值（AgentState）
# - next: 下一个要执行的节点列表
# - config: 状态配置（包含thread_id和时间戳）
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='518f2156-6cc3-4349-8f7d-4a12710c1c29'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in San Francisco"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '43a805b4-7ad2-41bd-9758-88d1e451e88f', 'token_usage': {'input_tokens': 1954, 'output_tokens': 22, 'total_tokens': 1976, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dca8932d-a32f-47cf-a0c6-1586a0f08cd0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'tool_call'}])]}, next=('action',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c20ca-be6f-6a66-8001-23204cb7b0eb'}}, metadata={'sourc

In [9]:
# 查看下一步操作
abot.graph.get_state(thread).next

('action',)

### continue after interrupt

In [10]:
# 继续执行（人工确认后）
# - None: 不提供新输入（继续执行）
# - thread: 指定会话
# 这告诉Agent：继续执行之前暂停的流程,相当于人工点击"确认执行"按钮
result = abot.graph.invoke(None, thread)

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'tool_call'}
Back to the model!
{'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='518f2156-6cc3-4349-8f7d-4a12710c1c29'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in San Francisco"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '43a805b4-7ad2-41bd-9758-88d1e451e88f', 'token_usage': {'input_tokens': 1954, 'output_tokens': 22, 'total_tokens': 1976, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dca8932d-a32f-47cf-a0c6-1586a0f08cd0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'tool_call'}]), ToolMessage(c

In [ ]:
# 检查当前状态
# get_state()返回：
# - values: 当前状态值（AgentState）
# - next: 下一个要执行的节点列表
# - config: 状态配置（包含thread_id和时间戳）
# state数据结构例子

# StateSnapshot(
#     values={  # 👉 这是真正的状态数据（包含初始值和所有中间值）
#         'messages': [
#             HumanMessage(content='Whats the weather in SF?'),
#             AIMessage(tool_calls=[...])
#         ]
#     },
#     next=('action',),  # 👉 下一步要执行的节点
#     config={  # 👉 状态的唯一标识（用于引用这个特定状态）
#         'configurable': {
#             'thread_id': '1',  # 会话ID
#             'checkpoint_id': '1f0c20ca-be6f-6a66-8001-23204cb7b0eb'  # 状态ID
#         }
#     },
#     metadata={  # 👉 元数据（额外信息）
#         'source': 'loop',  # 来源
#         'step': 1,  # 执行步骤计数
#         'parents': {}  # 父状态（用于状态历史链）
#     },
#     created_at='2025-11-15T10:20:12.792279+00:00',  # 创建时间
#     parent_config={  # 👉 指向父状态的链接（形成状态历史链）
#         'configurable': {
#             'thread_id': '1',
#             'checkpoint_id': '1f0c20ca-b0f2-64ff-8000-0f69f0da6573'
#         }
#     }
# )
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='518f2156-6cc3-4349-8f7d-4a12710c1c29'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in San Francisco"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '43a805b4-7ad2-41bd-9758-88d1e451e88f', 'token_usage': {'input_tokens': 1954, 'output_tokens': 22, 'total_tokens': 1976, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dca8932d-a32f-47cf-a0c6-1586a0f08cd0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_33e95959af8b46f5b1f05d', 'type': 'tool_call'}]), ToolMessage(content='{\'query\': \'current weather in San Francisco\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\

In [ ]:
# 查看下一步操作
abot.graph.get_state(thread).next

()

In [ ]:
# 示例使用2：发送消息并手动审批每个工具调用
messages = [HumanMessage("Whats the weather in LA?")]
thread2 = {"configurable": {"thread_id": "2"}}
result = abot.graph.invoke({"messages": messages}, thread2)

# 持续检查是否有下一步（即是否需要人工确认）
while abot.graph.get_state(thread2).next:
    print("\n", abot.graph.get_state(thread2),"\n")
    _input = input("proceed?")  # 人工确认，会出来一个输入框
    if _input != "y":
        print("aborting")
        break
    # 继续执行（人工确认后）
    result = abot.graph.invoke(None, thread2)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='b7723099-fa17-46de-a443-bb283e68a1d7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_bd7a36036b2643c6a9374c', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '90d276e2-bea8-41ce-822f-733d1d2cb4d1', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--4cc4a86b-684e-4ab0-8f7b-dd88156d4c56', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_bd7a36036b2643c6a9374c', 'type': 'tool_call'}])]}

 StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='b7723099-fa17-46de-a443-bb283e68a1d7'), AIMessage(content='', addit

## Modify State
Run until the interrupt and then modify the state.

运行直到中断，然后修改状态。

状态修改是指在AI Agent执行过程中，修改其记忆或决策历史的能力。就像你玩游戏时可以读取之前的存档，然后改变某些决定再继续游戏一样。

为什么需要状态修改？
纠正错误：如果Agent做出了错误决定，我们可以回到那个点并修正
人工干预：在关键决策前暂停，让人确认后再继续
调试分析：查看不同决策路径的结果
重试机制：当某步失败时，可以修改输入重新尝试

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread3 = {"configurable": {"thread_id": "4"}}
# 开启新的线程执行任务，会停在action之前
result = abot.graph.invoke({"messages": messages}, thread3)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}])]}


In [ ]:
# 获取当前状态，就是LLM后ACTION前的状态
current_values = abot.graph.get_state(thread3)
current_values.next

('action',)

In [ ]:
# LLM返回的内容
current_values.values['messages'][-1]

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}])

In [ ]:
# LLM返回的工具调用信息
current_values.values['messages'][-1].tool_calls

[{'name': 'tavily_search',
  'args': {'query': 'current weather in LA'},
  'id': 'call_72cde74b94974044a6fb0b',
  'type': 'tool_call'}]

In [ ]:
# 获取最后一个工具调用的ID（非常重要！保持ID不变才能正确替换）
_id = current_values.values['messages'][-1].tool_calls[0]['id']

# 修改工具调用参数：将查询地点从SF改为Louisiana
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search',
  'args': {'query': 'current weather in Louisiana'},
  'id': _id}
]

In [ ]:
# 新状态 - 这会创建一个新的状态快照
abot.graph.update_state(current_values.config, current_values.values)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_72cde74b94974044a6fb0b'}])]}


{'configurable': {'thread_id': '4',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0c20d6-f8b8-6944-8002-f0c823d61b84'}}

In [ ]:
# 从修改后的状态继续执行
abot.graph.get_state(thread3)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}])]}, next=('action',), config={'configurable': {'thread_id': '4', 'checkpoint_ns': '', 'checkpoint_id': '1f0c20d6-f8b8-6944-8002-f0c823d61b84'}}, metadata={'source': 'update', '

In [ ]:
# 继续运行
result = abot.graph.invoke(None, thread3)

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}
Back to the model!
{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}]), ToolMessage(content='{\'query\':

## Time Travel

In [ ]:
# 获取指定线程(thread)的所有状态历史记录
states = []
for state in abot.graph.get_state_history(thread3):
    print(state)
    print(state.next)
    print('--')
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}]), ToolMessage(content='{\'query\': \'current weather in Louisiana\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\': \'Weather in Lou

To fetch the same state as was filmed, the offset below is changed to `-3` from `-1`. This accounts for the initial state `__start__` and the first state that are now stored to state memory with the latest version of software.

为了获取与拍摄时相同的状态，下面的偏移量从 `-1` 改为 `-3`。这解释了初始状态 `__start__` 和第一个状态，它们现在已使用最新版本的软件存储到状态内存中。

In [ ]:
# 选择倒数第3个状态
# Python中负数索引表示从后往前数
# -1是最后一个（最新）状态，-2是倒数第二个，-3是倒数第三个
to_replay = states[-3]

In [27]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}])]}, next=('action',), config={'configurable': {'thread_id': '4', 'checkpoint_ns': '', 'checkpoint_id': '1f0c20d5-ce99-6cea-8001-e22acb389d23'}}, metadata={'source': 'loop', 'step': 1,

In [ ]:
# 回到过去并重新执行，从倒数第三个状态开始
result = abot.graph.invoke(None, to_replay.config)

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}
Back to the model!
{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}]), ToolMessage(content='{\'query\': \'current wea

## Go back in time and edit  回到过去并修改

In [ ]:
# 选择倒数第4个状态作为要修改的点
to_replay = states[-4]
to_replay.values['messages'][-1].tool_calls

[{'name': 'tavily_search',
  'args': {'query': 'current weather in Louisiana'},
  'id': 'call_72cde74b94974044a6fb0b',
  'type': 'tool_call'}]

In [ ]:
# 修改tool_calls内容
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search',
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [ ]:
# 将修改后的状态更新到状态历史中，创建一个新的状态分支
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)
print(to_replay)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA, accuweather'}, 'id': 'call_72cde74b94974044a6fb0b'}])]}
StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwar

In [ ]:
# 继续执行
result = abot.graph.invoke(None, branch_state)

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather in LA, accuweather'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}
Back to the model!
{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA, accuweather'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}]), ToolMessage(content=

## Add message to a state at a given time 在指定时间向某个状态添加消息

In [ ]:
# 选择倒数第3个状态作为要修改的点
to_replay=states[-3]
print(to_replay)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}])]}, next=('action',), config={'configurable': {'thread_id': '4', 'checkpoint_ns': '', 'checkpoint_id': '1f0c20d5-ce99-6cea-8001-e22acb389d23'}}, metadata={'source': 'loop', 'step': 1,

In [34]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
print(_id)

call_72cde74b94974044a6fb0b


In [ ]:
# 创建一个模拟的工具响应消息
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,  # 必须与原工具调用ID相同
    name="tavily_search",  # 工具名称（必须匹配）
    content="54 degree celcius"  # 人工提供的响应内容
)]}

In [ ]:
# 将人工响应添加到状态历史中
branch_and_add = abot.graph.update_state(
    to_replay.config,  # 使用原状态的配置（确保正确定位）
    state_update,      # 传入我们创建的人工响应
    as_node="action"   # 指定从哪个节点继续（这里是action节点之后）
)

In [ ]:
# 从branch_and_add指定的状态点继续执行
result = abot.graph.invoke(None, branch_and_add)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='eafaad4b-79b2-4e0e-a679-0d2de1820cd0'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'function', 'function': {'name': 'tavily_search', 'arguments': '{"query": "current weather in LA"}'}}]}, response_metadata={'finish_reason': 'tool_calls', 'request_id': '57c1efe5-3b4d-47ad-9c7b-8303a865f41e', 'token_usage': {'input_tokens': 1954, 'output_tokens': 21, 'total_tokens': 1975, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--dd032934-0dca-47e3-b371-cb053b46f14b', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in LA'}, 'id': 'call_72cde74b94974044a6fb0b', 'type': 'tool_call'}]), ToolMessage(content='{\'query\': \'current weather in LA\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'title\': \'Weather in Los Angeles\', \'url\': \'https://www.

# Extra Practice 额外练习

## Build a small graph
This is a small simple graph you can tinker with if you want more insight into controlling state memory.

这是一个简单的小图表，如果您想更深入了解如何控制状态记忆，可以对其进行修改。

In [73]:
from dotenv import load_dotenv

_ = load_dotenv()

In [74]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langgraph.checkpoint.sqlite import SqliteSaver

Define a simple 2 node graph with the following state:
- `lnode`: last node
- `scratch`: a scratchpad location
- `count` : a counter that is incremented each step

定义一个简单的二节点图，其状态如下：
- `lnode`：最后一个节点
- `scratch`：一个草稿纸位置
- `count`：一个计数器，每一步都会递增。

In [75]:
class AgentState(TypedDict):
    lnode: str
    scratch: str
    count: Annotated[int, operator.add]

In [76]:
def node1(state: AgentState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1",
            "count": 1,
           }
def node2(state: AgentState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2",
            "count": 1,
           }

The graph goes N1->N2->N1... but breaks after count reaches 3.

该图的走向为 N1->N2->N1...，但在计数达到 3 后中断。

In [77]:
def should_continue(state):
    return state["count"] < 3

In [78]:
builder = StateGraph(AgentState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)

builder.add_edge("Node1", "Node2")
builder.add_conditional_edges("Node2", 
                              should_continue, 
                              {True: "Node1", False: END})
builder.set_entry_point("Node1")

In [79]:
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)
graph = builder.compile(checkpointer=memory)

### Run it!
Now, set the thread and run!

In [80]:
thread = {"configurable": {"thread_id": str(1)}}
graph.invoke({"count":0, "scratch":"hi"},thread)

node1, count:0
node2, count:1
node1, count:2
node2, count:3


{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}

### Look at current state

Get the current state. Note the `values` which are the AgentState. Note the `config` and the `thread_ts`. You will be using those to refer to snapshots below.

获取当前状态。注意 `values`，它们代表 AgentState。注意 `config` 和 `thread_ts`。您将在下文中引用快照时用到它们。

In [81]:
graph.get_state(thread)

StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a6a-6dd1-8004-a2f64e695cb1'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-11-15T14:03:23.944699+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a69-6d56-8003-05dffe134153'}}, tasks=(), interrupts=())

View all the statesnapshots in memory. You can use the displayed `count` agentstate variable to help track what you see. Notice the most recent snapshots are returned by the iterator first. Also note that there is a handy `step` variable in the metadata that counts the number of steps in the graph execution. This is a bit detailed - but you can also notice that the *parent_config* is the *config* of the previous node. At initial startup, additional states are inserted into memory to create a parent. This is something to check when you branch or *time travel* below.

查看内存中的所有状态快照。您可以使用显示的 `count` agentstate 变量来帮助跟踪您看到的内容。请注意，迭代器会首先返回最新的快照。另请注意，元数据中有一个方便的 `step` 变量，用于统计图执行中的步骤数。这有点复杂——但您还可以注意到，`parent_config` 是前一个节点的 `config`。在初始启动时，会将额外的状态插入内存以创建父节点。当您进行分支或进行下面的“时间旅行”时，需要检查这一点。

### Look at state history

In [82]:
for state in graph.get_state_history(thread):
    print(state, "\n")

StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a6a-6dd1-8004-a2f64e695cb1'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-11-15T14:03:23.944699+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a69-6d56-8003-05dffe134153'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 3}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a69-6d56-8003-05dffe134153'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-11-15T14:03:23.944277+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a67-6417-8002-178a6493800f'}}, tasks=(PregelTask(id='2279386a-e42a-522b-2057-dc0ad47fc1ab', name='Node2', path=('__pregel_pull

Store just the `config` into an list. Note the sequence of counts on the right. `get_state_history` returns the most recent snapshots first.

将 `config` 存储到列表中。请注意右侧的计数顺序。`get_state_history` 首先返回最新的快照。

In [83]:
states = []
for state in graph.get_state_history(thread):
    states.append(state.config)
    print(state.config, state.values['count'])

{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a6a-6dd1-8004-a2f64e695cb1'}} 4
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a69-6d56-8003-05dffe134153'}} 3
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a67-6417-8002-178a6493800f'}} 2
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a64-669d-8001-b4f217cda102'}} 1
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a64-669c-8000-26ac60499518'}} 0
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a60-68d7-bfff-c3a5a72f8149'}} 0


Grab an early state.

In [84]:
states[-3]

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0c22bd-9a64-669d-8001-b4f217cda102'}}

This is the state after Node1 completed for the first time. Note `next` is `Node2`and `count` is 1.

这是 Node1 首次完成操作后的状态。注意，`next` 是 `Node2`，`count` 为 1。

In [85]:
graph.get_state(states[-3])

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 1}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a64-669d-8001-b4f217cda102'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-11-15T14:03:23.942057+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a64-669c-8000-26ac60499518'}}, tasks=(PregelTask(id='b8e111bc-9f0a-344e-18d5-4b08ea3eee8a', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result={'lnode': 'node_2', 'count': 1}),), interrupts=())

### Go Back in Time
Use that state in `invoke` to go back in time. Notice it uses states[-3] as *current_state* and continues to node2,

在 `invoke` 中使用该状态可以回溯到过去。请注意，它使用 states[-3] 作为 *current_state*，并继续执行到节点 2。

In [86]:
graph.invoke(None, states[-3])

node2, count:1
node1, count:2
node2, count:3


{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}

Notice the new states are now in state history. Notice the counts on the far right.

注意新的状态现在在状态历史中。注意最右侧的计数。

In [87]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state.config, state.values['count'])

{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618d-8004-d498e0e77117'}} 4
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618c-8003-7127451356df'}} 3
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618b-8002-a650bdf75780'}} 2
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a6a-6dd1-8004-a2f64e695cb1'}} 4
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a69-6d56-8003-05dffe134153'}} 3
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a67-6417-8002-178a6493800f'}} 2
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a64-669d-8001-b4f217cda102'}} 1
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bd-9a64-669c-8000-26ac60499518'}} 0
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkp

You can see the details below. Lots of text, but try to find the node that start the new branch. Notice the parent *config* is not the previous entry in the stack, but is the entry from state[-3].

您可以在下方看到详细信息。很多文本，但请尝试找到开始新分支的节点。请注意，父级 *config* 不是栈中的前一个条目，而是来自 state[-3] 的条目。

In [88]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state,"\n")

StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618d-8004-d498e0e77117'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-11-15T14:03:41.826701+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618c-8003-7127451356df'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 3}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618c-8003-7127451356df'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-11-15T14:03:41.826701+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-44f4-618b-8002-a650bdf75780'}}, tasks=(PregelTask(id='41691d8b-80bc-1141-9a66-5230ab15477e', name='Node2', path=('__pregel_pull

### Modify State
Let's start by starting a fresh thread and running to clean out history.

让我们从开启一个新线程开始，运行以清除历史记录。

In [89]:
thread2 = {"configurable": {"thread_id": str(2)}}
graph.invoke({"count":0, "scratch":"hi"},thread2)

node1, count:0
node2, count:1
node1, count:2
node2, count:3


{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}

In [90]:
from IPython.display import Image

# 图片方式执行失败，报缺少被依赖的dll包
# Image(graph.get_graph().draw_png())
graph.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | Node1 |    
  +-------+    
      .        
      .        
      .        
  +-------+    
  | Node2 |    
  +-------+    
      .        
      .        
      .        
 +---------+   
 | __end__ |   
 +---------+   


In [91]:
states2 = []
for state in graph.get_state_history(thread2):
    states2.append(state.config)
    print(state.config, state.values['count'])   

{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f7-8004-c30bde898f5b'}} 4
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f6-8003-afbd948be0f2'}} 3
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f5-8002-4128c226101f'}} 2
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f4-8001-5df773b28994'}} 1
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a8-6530-8000-4cd74143fdd6'}} 0
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a5-6c3c-bfff-cdd745a9968e'}} 0


Start by grabbing a state.

首先抓取一个状态。

In [92]:
save_state = graph.get_state(states2[-3])
save_state

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 1}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f4-8001-5df773b28994'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-11-15T14:03:49.975013+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a8-6530-8000-4cd74143fdd6'}}, tasks=(PregelTask(id='d2effd36-a90f-c4b6-3c40-57c069b2cd25', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result={'lnode': 'node_2', 'count': 1}),), interrupts=())

Now modify the values. One subtle item to note: Recall when agent state was defined, `count` used `operator.add` to indicate that values are *added* to the current value. Here, `-3` will be added to the current count value rather than replace it.

现在修改这些值。需要注意一点：回想一下，在定义代理状态时，`count` 使用 `operator.add` 来表示值是*添加到*当前值。这里，`-3` 会添加到当前的计数值，而不是替换它。

In [93]:
save_state.values["count"] = -3
save_state.values["scratch"] = "hello"
save_state

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': -3}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f4-8001-5df773b28994'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-11-15T14:03:49.975013+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a8-6530-8000-4cd74143fdd6'}}, tasks=(PregelTask(id='d2effd36-a90f-c4b6-3c40-57c069b2cd25', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result={'lnode': 'node_2', 'count': 1}),), interrupts=())

Now update the state. This creates a new entry at the *top*, or *latest* entry in memory. This will become the current state.

现在更新状态。这会在内存的*顶部*或*最新*位置创建一个新条目。这将成为当前状态。

In [94]:
graph.update_state(thread2,save_state.values)

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0c22bf-753f-6aec-8005-f0d0f04d9da3'}}

Current state is at the top. You can match the `thread_ts`.
Notice the `parent_config`, `thread_ts` of the new node - it is the previous node.

当前状态位于顶部。您可以匹配 `thread_ts`。
请注意新节点的 `parent_config` 和 `thread_ts`——它是前一个节点。

In [95]:
for i, state in enumerate(graph.get_state_history(thread2)):
    # if i >= 3:  #print latest 3
    #     break
    print(state.values)
    print(state.next)
    print(state, '\n')

{'lnode': 'node_1', 'scratch': 'hello', 'count': 1}
('Node1',)
StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': 1}, next=('Node1',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bf-753f-6aec-8005-f0d0f04d9da3'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2025-11-15T14:04:13.734372+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f7-8004-c30bde898f5b'}}, tasks=(PregelTask(id='d6792e44-66c1-7537-18fe-48e7909924bd', name='Node1', path=('__pregel_pull', 'Node1'), error=None, interrupts=(), state=None, result=None),), interrupts=()) 

{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}
()
StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22be-92a9-66f7-8004-c30bde898f5b'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}},

### Try again with `as_node`
When writing using `update_state()`, you want to define to the graph logic which node should be assumed as the writer. What this does is allow th graph logic to find the node on the graph. After writing the values, the `next()` value is computed by travesing the graph using the new state. In this case, the state we have was written by `Node1`. The graph can then compute the next state as being `Node2`. Note that in some graphs, this may involve going through conditional edges!  Let's try this out.

使用 `update_state()` 写入状态时，你需要向图逻辑指定哪个节点作为写入节点。这样做可以让图逻辑在图中找到该节点。写入值后，`next()` 的值会根据新的状态遍历图来计算。在本例中，当前状态是由 `Node1` 写入的。因此，图可以计算出下一个状态为 `Node2`。请注意，在某些图中，这可能涉及到条件边！让我们来尝试一下。

In [96]:
graph.update_state(thread2,save_state.values, as_node="Node1")

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0c22c0-16b8-67a2-8006-ecd0d0c80d54'}}

In [97]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': -2}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22c0-16b8-67a2-8006-ecd0d0c80d54'}}, metadata={'source': 'update', 'step': 6, 'parents': {}}, created_at='2025-11-15T14:04:30.665923+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bf-753f-6aec-8005-f0d0f04d9da3'}}, tasks=(PregelTask(id='29a1dd3f-a277-4d3c-8a6a-ca61e2869fbb', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result=None),), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': 1}, next=('Node1',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22bf-753f-6aec-8005-f0d0f04d9da3'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2025-11-15T14:04:13.734372+00:00', parent_config={'configurable': {'thread_id': '2', 'ch

`invoke` will run from the current state if not given a particular `thread_ts`. This is now the entry that was just added.

如果没有指定 `thread_ts`，`invoke` 将从当前状态运行。就是刚刚添加的条目。

In [98]:
graph.invoke(None,thread2)

node2, count:-2
node1, count:-1
node2, count:0
node1, count:1
node2, count:2


{'lnode': 'node_2', 'scratch': 'hello', 'count': 3}

Print out the state history, notice the `scratch` value change on the latest entries.

打印出状态历史记录，注意最新条目中的“scratch”值变化。

In [99]:
for state in graph.get_state_history(thread2):
    print(state,"\n")

StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hello', 'count': 3}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22c0-458a-66ce-800b-5b6d40534271'}}, metadata={'source': 'loop', 'step': 11, 'parents': {}}, created_at='2025-11-15T14:04:35.575367+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22c0-458a-66cd-800a-54e8293dc935'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': 2}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22c0-458a-66cd-800a-54e8293dc935'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2025-11-15T14:04:35.575367+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0c22c0-458a-66cc-8009-5fbe3edfa634'}}, tasks=(PregelTask(id='4acc09b3-8444-2920-c050-ebedb4e60252', name='Node2', path=('__pre

Continue to experiment!